# Designing a Voice Agent Platform

A voice platform coordinates live audio, model inference, conversation state, telephony, safety, and recovery for many simultaneous sessions.


## What to learn

- A session service owns call state and reconnect information.
- Streaming keeps recognition, reasoning, and speech generation responsive.
- Backpressure and admission control protect the system during load spikes.
- Provider adapters allow a failing speech or model service to be replaced or degraded.

## Decision rule

Design around a per-turn latency budget and explicit session state. Make reconnects idempotent, cap concurrency, isolate tenants, degrade nonessential features first, and measure cost per successful conversation minute.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Capacity + cost model for a 2,000-participant study launch.

Capacity planning asks: given provider quotas and a
study size, how many interviews can run concurrently, how long does the
study take, and what does it cost?
"""
# Import the dependencies used by this example.
import math

STUDY = {"participants": 2000, "interview_minutes": 30}

QUOTAS = {  # effective concurrent-session caps per provider account
    "deepgram_stt_streams": 150,
    "text_to_speech_sessions": 30,   # <-- the surprise bottleneck
    "llm_effective_sessions": 500,
    "worker_fleet_sessions": 1000,
}

COST_PER_MIN = {"stt": 0.005, "llm": 0.02, "tts": 0.05, "infra": 0.01}


# Define the data shape and small operations before running them.
def plan(study, quotas, tts_accounts=1):
    quotas = {**quotas,
              "text_to_speech_sessions": quotas["text_to_speech_sessions"] * tts_accounts}
    bottleneck = min(quotas, key=quotas.get)
    concurrency = quotas[bottleneck]
    waves = math.ceil(study["participants"] / concurrency)
    wall_clock_h = waves * study["interview_minutes"] / 60
    total_min = study["participants"] * study["interview_minutes"]
    cost = total_min * sum(COST_PER_MIN.values())
    return {"bottleneck": bottleneck, "concurrency": concurrency,
            "waves": waves, "wall_clock_hours": round(wall_clock_h, 1),
            "inference_cost_usd": round(cost)}


print("single TTS account:", plan(STUDY, QUOTAS))
print("sharded 5 TTS accounts:", plan(STUDY, QUOTAS, tts_accounts=5))
# Note the shape of the answer: sharding TTS moves the bottleneck to STT —
# capacity planning is iterative whack-a-mole across providers.


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
